# COSMOVERSE: End-to-End Tutorial (Stages 1–5 & Results 1–4)
### Hybrid Cosmological Modeling, Uncertainty Marginalization & Calibrated Inference Protocol

This notebook walks through all **3 Methodological Levels** and **4 Demonstrable Results** of [`docs/MASTER_DOCUMENT.md`](../docs/MASTER_DOCUMENT.md):
1. **Level 1 — Universe Physics**: Background cosmology, 1LPT + PM simulation, `TUNeSEmulator` (Result 1), and `BCEmuModel` + `BiasVarianceFrontier` (Result 2).
2. **Level 2 — Observational Realism (Before SBI)**: Survey masks, selection, Eulerian galaxy bias ($b_1, b_2$), and RSD multipoles ($P_0, P_2, P_4$).
3. **Level 3 — Calibrated Inference & Cross-Simulation Transfer**: `SBIEngine` (`ConditionalRealNVP`), `TARPCalibration` (Result 3), `DatasetPartitionProtocol` + `CAMELSHydroSpectraConnector` (Result 4), and `FieldLevelInferenceEngine` (Stage 5 / Paper 2).

In [ ]:
import numpy as np

from cosmoverse.baryons import BiasVarianceFrontier
from cosmoverse.core import CosmologicalParameters, Cosmology, SimulationParameters
from cosmoverse.datasets import CAMELSHydroSpectraConnector, build_camels_gen2_lh_dataloaders
from cosmoverse.emulators import TUNeSEmulator, evaluate_emulator_result1
from cosmoverse.inference import FieldLevelInferenceEngine
from cosmoverse.simulation import generate_gaussian_random_field

## 1. Result 1 — `TUNeSEmulator` Precision in Gaussian & Non-Gaussian 3D Statistics

In [ ]:
cosmo = Cosmology(CosmologicalParameters.planck2018())
sim_params = SimulationParameters(box_size=100.0, n_mesh=16, seed=42)
delta_lin, _ = generate_gaussian_random_field(cosmo, sim_params, seed=42)
target_field = delta_lin + 0.015 * (delta_lin**2 - np.mean(delta_lin**2))

tunes = TUNeSEmulator(base_features=8, depth=2, seed=42)
tunes.calibrate_to_simulation_bank(delta_lin[np.newaxis, ...], target_field[np.newaxis, ...])
pred_field = tunes.forward(delta_lin, scale_a=1.0, cosmo_params=cosmo.params)

res1 = evaluate_emulator_result1(pred_field, target_field, box_size=100.0, k_max=1.0)
print(f"P(k) MAPE (k <= 1 h/Mpc): {res1['pk_mape_pct']:.4f}% (Gate <1%: {res1['passes_pk_threshold']})")
print(f"3D Minkowski MAE: V0={res1['minkowski_mae']['V0']:.5f}, V3={res1['minkowski_mae']['V3']:.5f}")

## 2. Result 2 — Complete Baryonic & Observational Bias–Variance Frontier

In [ ]:
frontier = BiasVarianceFrontier()
res2 = frontier.compute_frontier_curve(k_max_values=[0.5, 1.0], marginalization_weights=[0.0, 0.5, 1.0])
opt = res2["optimal_operating_point"]
print(f"Optimal Point: k_max={opt['k_max']} h/Mpc, Bias={opt['bias_in_stage4_sigma']:.3f} sigma_S4, Inflation={opt['error_inflation_factor']:.2f}x")

## 3. Level 2 Observational Realism & Result 4 Multi-Suite Transfer (`IllustrisTNG`, `SIMBA`, `Astrid`)

In [ ]:
train_ds, test_ds, leak_ver = build_camels_gen2_lh_dataloaders(n_subset=16, grid_size=16, seed=42)
print(f"Leakage-Guarded 3D Loader: is_leakage_free={leak_ver['is_leakage_free']}")

connector = CAMELSHydroSpectraConnector()
for row in connector.evaluate_multi_suite_transfer_table(n_test_sims=15, seed=42):
    print(
        f"{row['target_suite']:20s} | Unmarg Bias={row['unmarginalized_bias_sigma']:.2f} sigma | "
        f"Marg Bias={row['marginalized_bias_sigma']:.2f} sigma | Cov68={row['post_transfer_cov68']*100:.1f}%"
    )

## 4. Stage 5 (Paper 2 Extension) — Differentiable Field-Level IC Reconstruction (`MCLMC` vs `HMC`)

In [ ]:
fl_engine = FieldLevelInferenceEngine(grid_size=6, box_size=100.0, noise_sigma=0.25)
fl_report = fl_engine.benchmark_mclmc_vs_hmc(n_samples=40, n_leapfrog=5, seed=42)
print(f"Large-Scale IC Cross-Correlation r(k): {fl_report['mean_large_scale_rk']:.4f}")
print(f"Preconditioned MCLMC EEVPD: {fl_report['preconditioned_mclmc']['eevpd']:.3e}")